# 📖 Bible Reading Progress Tracker — Pipeline Evaluation

**Purpose**  
Evaluate the NER pipeline on a representative sample before running on the full 16k dataset.

**Sampling strategy**  
Select 7 days where the dominant book changes — ensures coverage of different book names,
abbreviation styles, and chapter ranges without redundant repetition of the same pattern.

| # | Section | Description |
|---|---------|-------------|
| 0 | **Setup** | Imports, paths, orchestrator init |
| 1 | **Sample Selection** | Find 7 days where dominant book changes |
| 2 | **Extract** | Run NER extraction + validate output |
| 3 | **Persist** | Write to DB + sanity check row counts |
| 4 | **Schedule** | Validate reading plan for sampled dates |
| 5 | **Summarize** | Print daily compliance per sampled date |

## 0. Setup

In [1]:
import sys
import warnings
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
root = Path('..').resolve() # project root
sys.path.append(str(root / 'src'))

from bible_data import load_bible_data
from config.settings import PROCESSED_DIR, MODEL_PATH, READING_PLAN_PATH
from sessions.database import get_session, create_tables
from sessions.database import Member, Message, BibleReference, ReadingProgress
from pipelines import BibleProgressPipeline

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR = PROCESSED_DIR / 'pipeline_evaluation'
OUT_DIR.mkdir(parents=True, exist_ok=True)

bible_data = load_bible_data()

print('Setup complete.')

c:\one one\Desktop\bible_reading_recap_nlp\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


In [2]:
bible_pipeline = BibleProgressPipeline(
    saved_path=MODEL_PATH,
    bible_books=bible_data['books'],
    plan_path=READING_PLAN_PATH
)
print('Pipeline ready.')
print(f'  Reading Plan : {bible_pipeline.schedule.total_days} days ({bible_pipeline.schedule.dates[0]} → {bible_pipeline.schedule.dates[-1]})')

2026-04-07 12:43:09 | INFO     | bible_pipeline.extraction.extractor | Loading model from saved path: C:\one one\Desktop\bible_reading_recap_nlp\models\indobert-bible-ner-v6


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 460.52it/s, Materializing param=classifier.weight]                                      


Pipeline ready.
  Reading Plan : 7 days (2020-08-03 → 2020-08-09)


---
## 1. Sample Selection

Find 7 days where the dominant book changes — spread across the full date range.  
**Dominant book** = the book that appears most in progress messages on that day.

In [3]:
df = pd.read_csv(PROCESSED_DIR / 'reviewed_messages.csv')

df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['date'] = df['timestamp'].dt.date
df['is_progress'] = df['category'].str.strip().str.lower() == 'report'

progress_df = df[df['is_progress']].copy().reset_index(drop=True)

print(f'Total parsed messages: {len(progress_df):,}')
print(f'Date range: {progress_df["date"].min()} → {progress_df["date"].max()}')

Total parsed messages: 16,539
Date range: 2020-08-03 → 2022-06-25


In [4]:
from datetime import date

START_DATE = date(2020, 8, 3)
END_DATE = date(2020, 8, 9)

sample_df = progress_df[
    (progress_df['date'] >= START_DATE) &
    (progress_df['date'] <= END_DATE)
].copy()

print(f'Sampled messages : {len(sample_df)}')
print(f'Across {sample_df['date'].nunique()} days')
print(f'By {sample_df['sender'].nunique()} unique senders')
print()
sample_df.groupby('date')['message'].count().rename('message_count')

Sampled messages : 257
Across 7 days
By 47 unique senders



date
2020-08-03    40
2020-08-04    46
2020-08-05    42
2020-08-06    40
2020-08-07    42
2020-08-08    40
2020-08-09     7
Name: message_count, dtype: int64

---
## 2. Extract

Run NER extraction over the sampled messages using `BibleReferenceExtractor`.
Validation checks:
- **Zero refs** — progress messages that returned nothing
- **Invalid refs** — references where normalization flagged `is_valid=False`
- **Summary table** — overall extraction stats + resolver method breakdown
- **Spot-check** — verbose single-message debug

In [5]:
tqdm.pandas()

sample_df['ner_references'] = sample_df['message'].progress_apply(bible_pipeline.extractor.extract)
sample_df['ner_ref_count']  = sample_df['ner_references'].apply(len)

print(f'Done. {sample_df["ner_ref_count"].sum():,} total references extracted.')

100%|██████████| 257/257 [00:04<00:00, 54.02it/s]

Done. 273 total references extracted.


In [6]:
messages = sample_df['message'].tolist()

all_refs = bible_pipeline.extractor.extract_batch(messages)

---
### 2.1 Zero References

In [7]:
zero_refs = sample_df[sample_df['ner_ref_count'] == 0]

print(f'Messages with zero refs : {len(zero_refs):,} ({len(zero_refs) / len(sample_df) * 100:.1f}%)')
print()
zero_refs[['date', 'sender', 'message']].head(20)

Messages with zero refs : 0 (0.0%)



,date,sender,message


### 2.2 Invalid References

In [8]:
# Invalid refs
invalid_rows = sample_df[
    sample_df['ner_references'].apply(
        lambda refs: any(not r['is_valid'] for r in refs)
    )
]

print(f'Messages with invalid refs : {len(invalid_rows):,}')
for _, row in invalid_rows.head(10).iterrows():
    invalid = [r for r in row['ner_references'] if not r['is_valid']]
    print(f"  {row['message']!r:40s} → {invalid}")

Messages with invalid refs : 0


### 2.3 Extraction Summary

In [9]:
# Summary table
total = len(sample_df)
has_refs = (sample_df['ner_ref_count'] > 0).sum()
no_refs = (sample_df['ner_ref_count'] == 0).sum()
n_invalid = len(invalid_rows)


summary = pd.DataFrame([
    ('Total sampled', total, ''),
    ('References extracted', has_refs, f'{has_refs/total*100:.1f}%'),
    ('Zero references', no_refs, f'{no_refs/total*100:.1f}%'),
    ('Invalid references', n_invalid, f'{n_invalid/total*100:.1f}%'),
], columns=['Metric', 'Count', 'Pct'])

print(summary.to_string(index=False))

print('\nResolver stats:')
for method, count in bible_pipeline.get_extractor_stats().items():
    print(f'  {str(method):25s}: {count}')

              Metric  Count    Pct
       Total sampled    257       
References extracted    257 100.0%
     Zero references      0   0.0%
  Invalid references      0   0.0%

Resolver stats:
  exact                    : 544
  fuzzy                    : 2
  failed                   : 0


---
### 2.5 Exploded Results

In [10]:
refs_df = (
    sample_df[sample_df['ner_ref_count'] > 0]
    [['date', 'timestamp', 'sender', 'message', 'ner_references']]
    .explode('ner_references')
    .reset_index(drop=True)
)

refs_df = pd.concat([
    refs_df[['date', 'timestamp', 'sender', 'message']],
    refs_df['ner_references'].apply(pd.Series)
], axis=1)

print(f'Shape : {refs_df.shape}')
refs_df.head(10)

Shape : (273, 9)


,date,timestamp,sender,message,book_start,start_chapter,book_end,end_chapter,is_valid
0,2020-08-03,2020-08-03 03:48:00,Melanie Chandra,Kej 1-2 done,Kejadian,1,Kejadian,2,True
1,2020-08-03,2020-08-03 04:03:00,Lindawati Haryanto,Kej 1-2 done,Kejadian,1,Kejadian,2,True
2,2020-08-03,2020-08-03 04:08:00,Sherly Cahyadi,Kej 1-2 done,Kejadian,1,Kejadian,2,True
3,2020-08-03,2020-08-03 04:32:00,Seto Ninik,Kej 1-2 done,Kejadian,1,Kejadian,2,True
4,2020-08-03,2020-08-03 05:45:00,🪸Martha 🍁,Kej 1-2 done,Kejadian,1,Kejadian,2,True
5,2020-08-03,2020-08-03 06:07:00,Dewi Pratiwi,Kej 1-2 done,Kejadian,1,Kejadian,2,True
6,2020-08-03,2020-08-03 06:09:00,Endang Surati,Kej 1- 2 selesai.🙏,Kejadian,1,Kejadian,2,True
7,2020-08-03,2020-08-03 06:14:00,Dicky Andrian,Kej 1-2 done,Kejadian,1,Kejadian,2,True
8,2020-08-03,2020-08-03 06:14:00,🎍,Kej 1-2 done,Kejadian,1,Kejadian,2,True
9,2020-08-03,2020-08-03 06:15:00,"dr. Andreas C.N., Sp.B.",Kej 1-2 selesai,Kejadian,1,Kejadian,2,True


In [11]:
sample_df.to_csv(OUT_DIR / 'pipeline_sample.csv', index=False, encoding='utf-8-sig')
refs_df.to_csv(OUT_DIR / 'extracted_references.csv', index=False, encoding='utf-8-sig')
zero_refs[['date', 'sender', 'message']].to_csv(
    OUT_DIR / 'zero_refs.csv', index=False, encoding='utf-8-sig'
)

print(f'Saved pipeline_sample.csv      → {len(sample_df):,} rows')
print(f'Saved extracted_references.csv → {len(refs_df):,} rows')
print(f'Saved zero_refs.csv            → {len(zero_refs):,} rows')

Saved pipeline_sample.csv      → 257 rows
Saved extracted_references.csv → 273 rows
Saved zero_refs.csv            → 0 rows


---
## 3. Persist

Write extracted references to DB via `process_batch`. Sanity check: row counts per table should reflect what was inserted.

In [12]:
create_tables()
print('Tables ready.')

Tables created at: C:\one one\Desktop\bible_reading_recap_nlp\data\bible_progress.db
Tables ready.


In [13]:
totals = bible_pipeline.process_batch(sample_df)

print(f'Inserted {totals["refs"]:,} references')
print(f'Inserted {totals["chapters"]:,} chapter progress rows')
print(f'Skipped  {totals["skipped"]:,} invalid references')

Persisting: 100%|██████████| 257/257 [00:02<00:00, 90.09it/s]

2026-04-07 12:43:36 | INFO     | bible_pipeline.pipelines | Batch complete — refs=273 chapters=599 skipped=0
Inserted 273 references
Inserted 599 chapter progress rows
Skipped  0 invalid references


### 3.1 DB Row Count Validation

In [14]:
with get_session() as session:
    n_members  = session.query(Member).count()
    n_messages = session.query(Message).count()
    n_refs     = session.query(BibleReference).count()
    n_progress = session.query(ReadingProgress).count()

print(f'Members         : {n_members:,}')
print(f'Messages        : {n_messages:,}')
print(f'BibleReferences : {n_refs:,}')
print(f'ReadingProgress : {n_progress:,}')

# Sanity assertions
assert n_members  > 0, 'No members inserted'
assert n_messages > 0, 'No messages inserted'
assert n_refs     > 0, 'No references inserted'
assert n_progress > 0, 'No reading progress inserted'

# Cross-check: progress rows >= refs (each ref expands to >= 1 chapter)
assert n_progress >= n_refs, f'Progress rows ({n_progress}) < refs ({n_refs}) — unexpected'

print('\n✅ All DB sanity checks passed.')

Members         : 47
Messages        : 257
BibleReferences : 273
ReadingProgress : 556

✅ All DB sanity checks passed.


### 3.2 Spot-check — Sample Inserted Rows

In [15]:
with get_session() as session:
    sample_progress = (
        session.query(ReadingProgress)
        .limit(10)
        .all()
    )
    for row in sample_progress:
        print(row)

<ReadingProgress member_id=1 Kejadian ch.1
<ReadingProgress member_id=1 Kejadian ch.2
<ReadingProgress member_id=2 Kejadian ch.1
<ReadingProgress member_id=2 Kejadian ch.2
<ReadingProgress member_id=3 Kejadian ch.1
<ReadingProgress member_id=3 Kejadian ch.2
<ReadingProgress member_id=4 Kejadian ch.1
<ReadingProgress member_id=4 Kejadian ch.2
<ReadingProgress member_id=5 Kejadian ch.1
<ReadingProgress member_id=5 Kejadian ch.2


---
## 4. Schedule

Validate the reading plan loaded correctly for the sampled dates.
Checks:
- Each sampled date has assigned chapters
- Cross-book days expand correctly
- Emoji loaded per date

In [16]:
print('Reading plan validation\n')
for d in sorted(sample_df['date'].unique()):
    assigned = bible_pipeline.schedule.get_by_date(d)
    emoji = bible_pipeline.schedule.get_emoji(d)
    if not assigned:
        print(f'  ⚠️  {d} — NO CHAPTERS ASSIGNED')
        continue
    books = sorted({book for book, _ in assigned})
    print(f'  {d}  {emoji}  {len(assigned):>3} chapters  books={books}')

Reading plan validation

  2020-08-03  💥    2 chapters  books=['Kejadian']
  2020-08-04  🐑    2 chapters  books=['Kejadian']
  2020-08-05  ⚡    2 chapters  books=['Kejadian']
  2020-08-06  🌊    2 chapters  books=['Kejadian']
  2020-08-07  🌈    2 chapters  books=['Kejadian']
  2020-08-08  🏯    2 chapters  books=['Kejadian']
  2020-08-09  🛡️    2 chapters  books=['Kejadian']


In [17]:
# Cross-book day deep-check — print every chapter for days spanning >1 book
print('Cross-book day expansion:\n')
for d in sorted(sample_df['date'].unique()):
    assigned = bible_pipeline.schedule.get_by_date(d)
    books = list(dict.fromkeys(book for book, _ in assigned))
    if len(books) > 1:
        print(f'{d}:')
        for book, ch in assigned:
            print(f'  {book} {ch}')
        print()

Cross-book day expansion:



---
## 5. Summarize

Print daily compliance for each sampled date.
Members who completed all assigned chapters for that day are marked with the day's emoji.

In [18]:
for d in sorted(sample_df['date'].unique()):
    print(f'\n── {d} ──')
    bible_pipeline.summarize(d)


── 2020-08-03 ──
📖 *Kej 1 - 2* 📖
1. Agnes 💥
2. Andrie HG 
3. BL katering 💥
4. Bento 💥
5. Ci Ina Paperku 
6. Darius Handoko 💥
7. Dewi Pratiwi 💥
8. Dicky Andrian 💥
9. Endang Surati 💥
10. Gracia 💥
11. Helen Fransiska Margarita 💥
12. Ibu Kartika 💥
13. Ivan teguh 💥
14. Ko Martin 
15. Kristin WIjaya Nusantara 💥
16. LF 
17. Lenny Pandjidharma 💥
18. Lidia 
19. Lindawati Haryanto 💥
20. Matthew Henry 2 💥
21. Melanie Chandra 💥
22. Mfitri 💥
23. Moses 💥
24. Nathanael Jeffry Teja 💥
25. Noah 💥
26. Nurcahaya Sihombing 
27. Oma Lisa 💥
28. Ruri Handoko 💥
29. Seto Ninik 💥
30. Sherly Cahyadi 💥
31. Sim Ay Tjan 💥
32. Tan 
33. Tejo Jayadi 💥
34. Tjunfebelyana 💥
35. Tom 💥
36. Villas Bakery 💥
37. Wesley and Mom😇🥰 💥
38. Wiwik 💥
39. Yoppie 💥
40. Yoshua 
41. Yozef Tjandra 💥
42. dr. Andreas C.N., Sp.B. 💥
43. no vi3ika 💥
44. piccer 💥
45. scarlet 💥
46. 🎍 💥
47. 🪸Martha 🍁 💥

── 2020-08-04 ──
📖 *Kej 3 - 4* 📖
1. Agnes 🐑
2. Andrie HG 🐑
3. BL katering 🐑
4. Bento 🐑
5. Ci Ina Paperku Kej 2
6. Darius Handoko 🐑
7. Dewi Pratiw

In [19]:
row = pd.Series({
    "date": pd.to_datetime("2020-08-12").date(),
    "timestamp": pd.to_datetime("2020-08-12 10:30:33").to_pydatetime(),
    "sender": "Sherly Cahyadi",
    "message": "Kej 18 done",
})

bible_pipeline.process(row)

{'refs': 1, 'chapters': 6, 'skipped': 0, 'resumed': 0}